In [ ]:
import omnipath as op
import decoupler as dc

import numpy as np
import liana as li
import anndata as ad

import seaborn as sns
import matplotlib.pyplot as plt
import plotnine as p9

import squidpy as sq
import os

import pandas as pd
import scanpy as sc

In [ ]:
# Create Anndata object
adata = sc.read_h5ad("/projects/CRC_xenium/xenium_data/merged_obj.h5ad")
coords_df = pd.read_csv("/projects/CRC_xenium/xenium_data/xenium_coords.csv", index_col=0)
coords_df = coords_df.reindex(adata.obs_names)

In [ ]:
# Add coordinates information
adata.obsm["spatial"] = coords_df[["x", "y"]].values
print(adata)

In [ ]:
# Add CN information
CN_info = pd.read_csv("/projects/CRC_xenium/xenium_data/CN_results/ALL_cells_r=50_k=20_CN=10.csv", index_col=0)
CN_info = CN_info.set_index('cell_id')
CN_info.index.name = None
adata.obs["All_CN_new"] = CN_info["neighborhood20"]

In [ ]:
# Align cell order between AnnData and coordinate df
coords_df = coords_df.reindex(adata.obs_names)

In [ ]:
adata.layers["counts"] = adata.X.copy()

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
# Extract cell to cell interaction information
ligrec = op.interactions.import_intercell_network()
ligrec = ligrec.rename(columns={'genesymbol_intercell_source':'ligand', 'genesymbol_intercell_target':'receptor'})
ligrec = ligrec[['ligand', 'receptor', 'references'] + [col for col in ligrec.columns if col not in ['ligand', 'receptor', 'references']]]
ligrec = ligrec[ligrec["references"].str.contains("CellPhoneDB|CellChatDB|connectomeDB2020", na=False)]
ligrec = ligrec[ligrec["consensus_direction"] == True]

In [ ]:
##### Search for a bandwidth that gives approximately 20 neighbors and compute the inflow score for each sample #####

sample_ids = adata.obs["sample_id"].unique().tolist()

results = {}
bw_summary = []
lrdata_list = []

for sid in sample_ids:
    print(f"\nProcessing: {sid}")
    
    # 1. Subset the data by sample
    adata_sub = adata[adata.obs["sample_id"] == sid].copy()
    
    # Check whether spatial coordinates are available
    if "spatial" not in adata_sub.obsm:
        print(f"  Skipped {sid}: 'spatial' not found in obsm")
        continue
    
    # Skip samples with too few cells, if necessary
    if adata_sub.n_obs < 20:
        print(f"  Skipped {sid}: too few cells ({adata_sub.n_obs})")
        continue

    try:
        # 2. Search for a bandwidth that gives approximately 20 neighbors
        plot, df = li.ut.query_bandwidth(
            coordinates=adata_sub.obsm["spatial"],
            start=5,
            end=100,
            interval_n=40
        )

        # 3. Select the bandwidth with the number of neighbors closest to 20
        idx = (df["neighbours"] - 20).abs().idxmin()
        bw = df.loc[idx, "bandwith"]

        # 4. Construct the spatial neighbor graph
        li.ut.spatial_neighbors(
            adata=adata_sub,
            bandwidth=bw,
            spatial_key="spatial"
        )

        # 5. Compute inflow score
        lrdata = li.mt.inflow(
            adata_sub,
            groupby="sub_type",
            resource=ligrec[["ligand", "receptor"]],
            use_raw=False,
            verbose=True
        )

        # 6. Compute global specificity
        li.mt.compute_global_specificity(
            lrdata,
            groupby="sub_type",
            use_raw=False,
            verbose=True
        )

        # 7. Save results
        results[sid] = {
            "query_plot": plot,
            "query_df": df,
            "bandwidth": bw,
            "lrdata": lrdata,
        }

        bw_summary.append({
            "sample_id": sid,
            "n_cells": adata_sub.n_obs,
            "bandwidth": bw,
            "mean_neighbours": df.loc[idx, "neighbours"]
        })

        lrdata.obs["sample_id"] = sid
        lrdata_list.append(lrdata)

        print(f"  Done: {sid} | bw={bw} | mean_neighbours={df.loc[idx, 'neighbours']}")

    except Exception as e:
        print(f"  Failed: {sid} | {e}")

In [ ]:
# Merge results
lrdata_merged = ad.concat(
    lrdata_list,
    join="outer",
    label="sample_id",
    keys=sample_ids,
    index_unique=None
)

In [ ]:
# Code for visualizing ligand–receptor interactions between specific cell types

# 1. Specify the column containing cell-type annotations
obs_col_name = 'sub_type' 

# 2. Specify the target cell type
target_cells = ['Endothelial/Mural']

# 3. Create a mask that is True only for cells matching the target cell type
mask = lrdata_merged.obs[obs_col_name].isin(target_cells)

# 4. Extract the original values of the selected ligand–receptor interaction
original_values = lrdata_merged.obs_vector('MGP_high_Fib^VEGFA^FLT1')

# 5. Create a new column in .obs and initialize all values to 0
lrdata_merged.obs['MGP_Fib^VEGFA^FLT1_Endo'] = 0.0

# 6. Assign the original values only to the target cells where the mask is True
lrdata_merged.obs.loc[mask, 'MGP_Fib^VEGFA^FLT1_Endo'] = original_values[mask]

# Check the results
print(lrdata_merged.obs[[obs_col_name, 'MGP_Fib^VEGFA^FLT1_Endo']].head())

In [ ]:
MSI_adata_tumor = lrdata_merged[
    (lrdata_merged.obs["slide_id"] == "Slide_1") &
    (lrdata_merged.obs["Phenotype"] == "Tumor")
].copy()
MSS_adata_tumor = lrdata_merged[
    (lrdata_merged.obs["slide_id"] == "Slide_2") &
    (lrdata_merged.obs["Phenotype"] == "Tumor")
].copy()
MSI_adata_border = lrdata_merged[
    (lrdata_merged.obs["slide_id"] == "Slide_1") &
    (lrdata_merged.obs["Phenotype"] == "border")
].copy()
MSS_adata_border = lrdata_merged[
    (lrdata_merged.obs["slide_id"] == "Slide_2") &
    (lrdata_merged.obs["Phenotype"] == "border")
].copy()

In [ ]:
# --- Generate the plot ---
spatial_coords = MSI_adata_border.obsm["spatial"]
rotated_coords = np.column_stack((spatial_coords[:, 1], -spatial_coords[:, 0]))
MSI_adata_border.obsm["spatial_rotated"] = rotated_coords

fig, ax = plt.subplots(figsize=(10, 45))

sc.pl.embedding(
    MSI_adata_border,
    basis="spatial_rotated",
    color="MGP_Fib^VEGFA^FLT1_Endo",
    s=50,
    cmap="magma_r", 
    ax=ax,
    show=False
)

plt.show()

In [ ]:
# --- Generate the plot ---
spatial_coords = MSS_adata_border.obsm["spatial"]
rotated_coords = np.column_stack((spatial_coords[:, 1], -spatial_coords[:, 0]))
MSS_adata_border.obsm["spatial_rotated"] = rotated_coords

fig, ax = plt.subplots(figsize=(10, 45)) 
 
sc.pl.embedding(
    MSS_adata_border,
    basis="spatial_rotated",
    color="MGP_Fib^VEGFA^FLT1_Endo",
    s=50,
    cmap=plt.colormaps()[90], 
    ax=ax,
    show=False
)

plt.show()

In [ ]:
# --- Generate the plot ---
spatial_coords = MSS_adata_tumor.obsm["spatial"]
rotated_coords = np.column_stack((spatial_coords[:, 1], -spatial_coords[:, 0]))
MSS_adata_tumor.obsm["spatial_rotated"] = rotated_coords

fig, ax = plt.subplots(figsize=(3,3)) 
 
sc.pl.embedding(
    MSS_adata_tumor,
    basis="spatial_rotated",
    color="MGP_Fib^VEGFA^FLT1_Endo", 
    s=50,
    cmap=plt.colormaps()[90], 
    ax=ax,
    show=False
)

plt.show()

In [ ]:
# --- Generate the plot ---
spatial_coords = MSI_adata_tumor.obsm["spatial"]
rotated_coords = np.column_stack((spatial_coords[:, 1], -spatial_coords[:, 0]))
MSI_adata_tumor.obsm["spatial_rotated"] = rotated_coords

fig, ax = plt.subplots(figsize=(10,45))
 
sc.pl.embedding(
    MSI_adata_tumor,
    basis="spatial_rotated",
    color="MGP_Fib^VEGFA^FLT1_Endo",
    s=50,
    cmap=plt.colormaps()[90],
    vmax=3,
    ax=ax,
    show=False
)

plt.show()